# stride-zero-broadcast — worked example 3: contiguous() materializes a zero-stride view

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `stride-zero-broadcast`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Calling `.contiguous()` on a zero-stride broadcast view allocates fresh memory and writes a real copy with normal (nonzero) strides. After materialization the result no longer shares storage with the source and `.is_contiguous()` is True.

## Worked solution

We expand a `(3,)` vector to `(2, 3)` (stride `(0, 1)`, not contiguous) and then call `.contiguous()`. The materialized tensor has stride `(3, 1)` — no zero — and a different `data_ptr()` from the source, proving a copy occurred. We print `is_contiguous()` before (False) and after (True), and confirm the values are unchanged. This is how you convert a cheap broadcast view into an independent buffer.

In [ ]:
x = t.tensor([1.0, 2.0, 3.0])
view = x.expand(2, 3)
mat = view.contiguous()
print('view contiguous:', view.is_contiguous())
print('mat contiguous:', mat.is_contiguous())
print('view stride:', tuple(view.stride()), 'mat stride:', tuple(mat.stride()))
print('mat is a copy:', mat.data_ptr() != x.data_ptr())
print('values preserved:', t.equal(view, mat))